<a href="https://colab.research.google.com/github/Nourimanekerroumi996/olist-pyspark/blob/setup/otebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Olist – Mise en place de l'environnement

Dans ce notebook, je prépare tout ce qu'il faut pour travailler sur le dataset e-commerce **Olist**:
installation des outils, récupération des données depuis Kaggle, démarrage de Spark et chargement des 9 tables.

## 1. Installation des outils

In [2]:
!pip install pyspark kaggle -q

## 2. Connexion à Kaggle


In [5]:
from google.colab import files
import os

# Kaggle va chercher sa clé
# et si elle n'y est pas encore, on la demande. Sinon, on ne redemande rien.
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    print("Veuillez uploader votre fichier kaggle.json")
    files.upload()

    # On range la clé à sa place...
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/

    # et on la rend privée
    !chmod 600 ~/.kaggle/kaggle.json
else:
    print("La clé Kaggle est déjà en place ")

Veuillez uploader votre fichier kaggle.json


Saving kaggle.json to kaggle.json


3. Téléchargement et dézippage du dataset Olist

In [6]:
# On récupère le dataset Olist  puis on le décompresse dans olist_data/
!kaggle datasets download -d olistbr/brazilian-ecommerce
!unzip -o -q brazilian-ecommerce.zip -d olist_data

# Petit coup d'œil pour vérifier qu'on a bien nos 9 fichiers CSV
!ls -lh olist_data

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
100% 42.6M/42.6M [00:00<00:00, 154MB/s]

total 121M
-rw-r--r-- 1 root root 8.7M Oct  1  2021 olist_customers_dataset.csv
-rw-r--r-- 1 root root  59M Oct  1  2021 olist_geolocation_dataset.csv
-rw-r--r-- 1 root root  15M Oct  1  2021 olist_order_items_dataset.csv
-rw-r--r-- 1 root root 5.6M Oct  1  2021 olist_order_payments_dataset.csv
-rw-r--r-- 1 root root  14M Oct  1  2021 olist_order_reviews_dataset.csv
-rw-r--r-- 1 root root  17M Oct  1  2021 olist_orders_dataset.csv
-rw-r--r-- 1 root root 2.3M Oct  1  2021 olist_products_dataset.csv
-rw-r--r-- 1 root root 171K Oct  1  2021 olist_sellers_dataset.csv
-rw-r--r-- 1 root root 2.6K Oct  1  2021 product_category_name_translation.csv


4. Initialisation de la Session Spark

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (SparkSession.builder
         .appName("Olist_Intensive_Training")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

print("Spark est prêt  version :", spark.version)

Spark est prêt  version : 4.0.4


## 5. Chargement des tables
Chaque CSV est chargé dans un DataFrame Spark et rangé dans un dictionnaire

In [9]:
datasets = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv"
]

dfs = {}
for dataset in datasets:
    name = dataset.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
    path = f"olist_data/{dataset}"
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True)
    print(f"Dataset '{name}' chargé. Lignes : {dfs[name].count()}")

# Exemple : Accès au DataFrame des commandes
# dfs['orders'].show(5)

Dataset 'customers' chargé. Lignes : 99441
Dataset 'geolocation' chargé. Lignes : 1000163
Dataset 'order_items' chargé. Lignes : 112650
Dataset 'order_payments' chargé. Lignes : 103886
Dataset 'order_reviews' chargé. Lignes : 104162
Dataset 'orders' chargé. Lignes : 99441
Dataset 'products' chargé. Lignes : 32951
Dataset 'sellers' chargé. Lignes : 3095
Dataset 'product_category_name_translation' chargé. Lignes : 71


## 6. Premier aperçu

In [10]:
# On jette un œil à la table centrale du projet : les commandes
dfs["orders"].printSchema()        # les colonnes et leur type
dfs["orders"].show(5, truncate=False)  # les 5 premières lignes, sans couper le texte

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------